# 05 · Airport archetypes

**RQ3.** Do U.S. airports fall into distinct delay-risk archetypes once size and
cause mix are considered together?

Clustering is reported with its selection evidence — a silhouette sweep, a Ward
cross-check and a subsample stability score — because a k-means solution that
changes when a third of the airports are removed describes the sample, not the
airports.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

In [ ]:
from team01 import cluster, features, viz

panel = clean.completed(io.load_flights())
airports = io.load_airports()

matrix = features.airport_feature_matrix(panel, airports, min_departures=2000)
print(f"{len(matrix)} airports x {matrix.shape[1]} columns")
matrix.head(3)

In [ ]:
sweep = cluster.sweep_k(matrix)
chosen = cluster.choose_k(sweep)
print(f"chosen k = {chosen}")
sweep

In [ ]:
labelled, _, _ = cluster.fit(matrix, chosen)
names = cluster.name_clusters(labelled)
labelled["archetype"] = labelled["cluster"].map(names)

print(f"Ward cophenetic correlation {cluster.ward_check(matrix)['cophenetic_correlation']}")
print(cluster.stability(matrix, chosen))

cluster.cluster_table(labelled)[["cluster", "archetype", "airports",
                                 "mean_arr_delay", "share_delayed_60",
                                 "largest_members"]]

In [ ]:
viz.fig_silhouette(sweep, chosen)
viz.fig_cluster_scatter(labelled, names)